# Lab 06: training workflow

Seed **2473986**. This workflow is unexecuted in this package. The completed Colab run used the same CLI. Its measured outputs are in `results/` and `Review_Results.ipynb`. Use a new output directory when reproducing training.


In [ ]:
import sys, subprocess, json
from pathlib import Path
ROOT = Path.cwd()
assert (ROOT / "lab06.py").is_file(), "Open the notebook from the Lab06_Ready folder."
print(sys.executable)


## 1. Install dependencies

First run requires Internet for packages and pretrained weights. Skip this cell when the correct environment is already installed.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")], check=True)


## 2. Select local dataset

Select the directory directly containing metadata.jsonl, good/ and bad/. The images do not need to be uploaded. The original dataset metadata will not be overwritten.

In [ ]:
# Edit these paths for your machine or Colab session.
DATASET = Path("/content/lab06_data/demo_quality_dataset")
OUT = Path("/content/Lab06_new_results")
assert all((DATASET / x).exists() for x in ("metadata.jsonl", "good", "bad"))
assert not (OUT / "protocol.json").exists(), "Choose a new output folder."


## 3. Predeclare the experiment

The frozen protocol uses source-grouped 70/15/15 splits, whole RGB images at 224×224, seed 2473986, no extra augmentation and threshold 0.5. Provisional acceptance targets are engineering assumptions, not teacher-specified thresholds. The memo explains frozen-base transfer learning, partial unfreezing and full fine-tuning.

In [ ]:
print((ROOT / "config.json").read_text())
print((ROOT / "docs" / "Model_Selection_Memo.md").read_text())


## 4. Run all experiments

The runner trains MobileNetV2 and EfficientNetB0 frozen-base baselines and fine-tunes both YOLOv8 classification variants. It trains every candidate before final evaluation, uses validation for checkpoint and model selection, then measures test metrics at the fixed threshold. No test-driven tuning is permitted. It can resume completed stages after an interruption. Actual training may be slow on CPU.

In [ ]:
subprocess.run([sys.executable, "-u", str(ROOT / "lab06.py"), "run", "--dataset", str(DATASET), "--out", str(OUT)], check=True)


## 5. Training curves and test confusion matrices

Compare training and validation losses. Look for widening gaps and rising validation loss before attributing overfitting. Good/bad label order is explicit across frameworks.

In [ ]:
from IPython.display import display, Image, Markdown
for name in ("mobilenetv2", "efficientnetb0"):
    display(Markdown("### " + name))
    display(Image(filename=str(OUT / name / "curves.png")))
for name in ("yolov8n", "yolov8s"):
    display(Markdown("### " + name))
    display(Image(filename=str(OUT / name / "results.png")))
display(Image(filename=str(OUT / "confusion_matrices.png")))
display(Image(filename=str(OUT / "accuracy_vs_speed.png")))


## 6. Full comparison and decision

All metrics below are generated from real local outputs. Bad-class precision/recall/F1 accompany accuracy; batch-one CPU timing includes resize/preprocessing and forward inference but excludes disk image decoding. Read the scope and limitations in the final report. No candidate passing all targets means no deployment acceptance.

In [ ]:
display(Markdown((OUT / "Final_Report.md").read_text(encoding="utf-8")))
print((OUT / "comparison.csv").read_text())
print((OUT / "defect_analysis.csv").read_text())


## 7. Failure analysis and presentation

Inspect actual false accepts and false rejects. Do not assume glare is the hardest defect. A five-minute speaking outline is generated; the in-class delivery and repository commit still need to be performed.

In [ ]:
for name in ("mobilenetv2", "efficientnetb0", "yolov8n", "yolov8s"):
    display(Image(filename=str(OUT / name / "failure_gallery.png")))
display(Markdown((OUT / "Presentation_5min.md").read_text(encoding="utf-8")))
